# E5 (2022)
---
[[paper]](https://arxiv.org/pdf/2212.03522)<br>E5 = Embeddings for Everything

E5 — это семейство state-of-the-art моделей для создания текстовых эмбеддингов, разработанное Microsoft. Модели E5 базируются на энкодерах T5 и обучаются с использованием contrastive learning на очень большом и разнообразном наборе данных, что позволяет им генерировать высококачественные и универсальные эмбеддинги для широкого спектра задач.

### Контекст

Создание универсальных текстовых эмбеддингов, которые хорошо работают в различных задачах (семантический поиск, кластеризация, классификация, Retrieval-Augmented Generation), является давней целью. Существующие на момент появления E5 модели часто демонстрировали отличные результаты в одних задачах, но требовали дообучения или показывали более низкое качество в других, или же были проприетарными решениями (например, OpenAI Embeddings, 2022). Разработчики искали способ создать одну мощную модель, способную генерировать "эмбеддинги для всего", которая была бы при этом открытой и доступной.

### Идея

Основная идея E5 заключалась в создании *единой, высокопроизводительной модели* для получения dense embeddings, которая превосходила бы специализированные модели в широком спектре задач и не требовала бы дообучения под конкретный сценарий. Для этого было предложено несколько ключевых инноваций:
1.  **Базовый энкодер T5:** Использование мощного Transformer-энкодера из архитектуры T5 (Text-to-Text Transfer Transformer), известной своими способностями к пониманию и генерации текста, в качестве основы.
2.  **Масштабные и разнообразные данные:** Сбор и агрегация беспрецедентно большого и разнообразного набора данных для обучения, включающего миллионы пар (запрос, документ) из различных источников и доменов.
3.  **Специфическая методика обучения:** Применение contrastive learning с особым подходом к форматированию входных данных (`query:` для запросов и `passage:` для документов) и использованием mean pooling для агрегации выходных состояний энкодера.

### Постановка задачи

Модель E5 решает задачу создания низкоразмерных dense vector representations (эмбеддингов) для произвольных текстовых фрагментов (запросов, предложений, абзацев, документов). Цель состоит в том, чтобы семантически похожие тексты имели эмбеддинги, близкие друг к другу в векторном пространстве (обычно измеряется косинусным сходством или скалярным произведением). Эти эмбеддинги затем могут быть использованы для эффективного поиска, кластеризации или сравнения текстов.

### Альтернативные методы

На момент появления E5 существовали следующие основные подходы к генерации текстовых эмбеддингов:
*   **Sparse Retrieval (например, BM25):** Традиционные методы, основанные на частоте слов, которые хорошо работают для точных ключевых слов, но плохо улавливают семантическое сходство.
*   **Sentence-BERT (SBERT, 2019):** Пионерский подход к дообучению BERT-подобных моделей для создания эмбеддингов предложений с использованием Siamese/Triplet сетей. SBERT значительно улучшил качество эмбеддингов по сравнению с "сырыми" эмбеддингами от BERT (`[CLS]` токен), но часто использовал меньшие наборы данных для обучения и мог быть менее универсальным.
*   **Dense Passage Retrieval (DPR, 2020):** Модель с двухбашенной архитектурой на базе BERT, специально разработанная для retrieval-задач и обученная на QA-данных. Хорошо работала для информационного поиска, но могла быть менее общей для других задач.
*   **Другие модели на основе BERT/RoBERTa/ELECTRA:** Множество follow-up работ, использующих различные архитектуры BERT-семейства и обучающихся по схожим методикам.
*   **Проприетарные модели (например, OpenAI Embeddings, Cohere Embeddings):** Высокопроизводительные, но закрытые модели, детали архитектуры и обучения которых не публиковались.

### Архитектура

Модели E5 строятся на основе **энкодера T5** различных размеров: `e5-small`, `e5-base`, `e5-large`, `e5-xl`, `e5-2b`. Эти размеры соответствуют предобученным чекпойнтам T5 (например, `t5-small`, `t5-base`, `t5-large`, `t5-3b`, `t5-11b`).

Основные компоненты:
1.  **Входное форматирование:** Каждый входной текст (запрос или документ) получает префикс, который помогает модели различать их роль: `query: ` для запросов и `passage: ` для документов. Например, запрос "What is AI?" становится "query: What is AI?". Документ "Artificial Intelligence is..." становится "passage: Artificial Intelligence is...".
2.  **T5 Encoder:** Преобразованный текст подается на вход Transformer-энкодеру T5. T5, в отличие от BERT, изначально разработан для задач Text-to-Text, но его энкодер очень эффективен для извлечения контекстуальных представлений.
3.  **Pooling Layer:** После получения выходных скрытых состояний от энкодера, применяется **mean pooling** (усреднение) по всем токенам в последовательности (исключая padding токены). Это агрегирует контекстуальные представления каждого токена в один фиксированный вектор. В отличие от использования только `[CLS]` токена, mean pooling часто дает более стабильные и общие эмбеддинги для разнообразных задач.
4.  **L2 Normalization:** Полученный вектор эмбеддинга подвергается L2-нормализации, что является стандартной практикой для dense embeddings, особенно при использовании косинусного сходства.

### Алгоритм обучения

Обучение E5 основано на **contrastive learning**. Цель состоит в том, чтобы максимизировать сходство между эмбеддингами положительных пар (запрос, релевантный документ) и минимизировать сходство с отрицательными парами (запрос, нерелевантный документ).

1.  **Набор данных:** Обучение проводится на гигантском и разнообразном наборе данных, агрегированном из множества источников, включая:
    *   MS MARCO (passage и document ranking)
    *   Natural Questions
    *   HotpotQA
    *   InfoXLM
    *   и другие датасеты, содержащие пары (запрос, релевантный документ).
    Это обеспечивает широкий охват доменов и стилей текста.

2.  **Формирование батчей:** Для каждого батча берутся пары (запрос $Q_i$, положительный документ $D_i^+$). В качестве отрицательных примеров ($D_j^-$) используются:
    *   In-batch negatives: документы из того же батча, которые не являются положительными для текущего запроса.
    *   Иногда используются **hard negatives**: документы, которые семантически близки к запросу, но не являются релевантными, и которые модель ранее ошибочно считала положительными.

3.  **Функция потерь:** Используется InfoNCE loss (или его вариации на основе cross-entropy). Для каждого запроса $Q_i$ и его положительного документа $D_i^+$ функция потерь поощряет высокую оценку сходства $sim(Emb(Q_i), Emb(D_i^+))$ по сравнению со всеми отрицательными документами $D_j^-$ в батче.
    $$L = -\sum_i \log \frac{\exp(sim(Emb(Q_i), Emb(D_i^+)) / \tau)}{\sum_{j \in B} \exp(sim(Emb(Q_i), Emb(D_j)) / \tau)}$$
    Где $\tau$ — это температурный параметр, $B$ — множество всех документов в батче (включая $D_i^+$).

4.  **Шаги обучения:**
    *   Для каждого батча:
        1.  Преобразовать каждый запрос $Q_i$ в "query: $Q_i$".
        2.  Преобразовать каждый документ $D_j$ в "passage: $D_j$".
        3.  Пропустить все преобразованные тексты через T5-энкодер.
        4.  Применить mean pooling и L2-нормализацию для получения эмбеддингов $E_{Q_i}$ и $E_{D_j}$.
        5.  Вычислить скалярное произведение (или косинусное сходство) между эмбеддингом каждого запроса и эмбеддингами всех документов в батче.
        6.  Вычислить InfoNCE loss.
        7.  Выполнить обратное распространение ошибки и обновить веса модели.

### Алгоритм инференса

Процесс инференса для E5 прост и стандартизирован:

1.  **Создание эмбеддинга запроса:**
    *   Берется запрос $Q$.
    *   К нему добавляется префикс: `query: ` (например, "query: лучшие модели ИИ").
    *   Полученная строка подается на вход обученному энкодеру E5.
    *   Применяется mean pooling к выходным скрытым состояниям.
    *   Результат L2-нормализуется. Это и есть эмбеддинг запроса $E_Q$.

2.  **Создание эмбеддинга документа/пассажа:**
    *   Берется документ $D$.
    *   К нему добавляется префикс: `passage: ` (например, "passage: Модель E5 от Microsoft...").
    *   Полученная строка подается на вход обученному энкодеру E5.
    *   Применяется mean pooling к выходным скрытым состояниям.
    *   Результат L2-нормализуется. Это и есть эмбеддинг документа $E_D$.

3.  **Поиск сходства / Retrieval:**
    *   Эмбеддинги для всех документов в корпусе заранее генерируются (оффлайн) и сохраняются в специальном индексе для Approximate Nearest Neighbor (ANN) поиска (например, FAISS, Annoy).
    *   При поступлении нового запроса, генерируется его эмбеддинг $E_Q$.
    *   С помощью ANN-индекса находятся $K$ ближайших эмбеддингов документов к $E_Q$ (используя косинусное сходство или скалярное произведение).
    *   Возвращаются исходные документы, соответствующие этим эмбеддингам.

### Результаты

Модели E5 продемонстрировали выдающиеся результаты, часто устанавливая новые state-of-the-art показатели в широком спектре задач:

*   **MTEB Leaderboard:** E5 модели, особенно `e5-large` и `e5-2b`, показали исключительную производительность на Massive Text Embedding Benchmark (MTEB), который включает в себя 58 бенчмарков из 8 категорий (bitext mining, classification, clustering, pair classification, reranking, retrieval, semantic textual similarity, summarization). Они регулярно занимали первые места, подтверждая свою универсальность.
*   **Производительность Retrieval:** На ключевых бенчмарках информационного поиска, таких как MS MARCO Passage Ranking и BEIR, E5 превзошла предыдущие SOTA модели (например, DPR, SBERT-варианты) по метрикам nDCG@K и Recall@K. Например, на MS MARCO `e5-large` показала заметное улучшение в nDCG@10 по сравнению с предшественниками.
*   **Обобщающая способность:** Ключевое достижение E5 — это сильная производительность *без* дополнительного дообучения под конкретную задачу, что подтверждает концепцию "эмбеддингов для всего". Это значительно снижает барьер входа для использования dense retrieval в новых доменах.

## 📝 Критический анализ

```markdown
# E5 (2022)
---
[[paper]](https://arxiv.org/pdf/2212.03522)<br>E5 = Embeddings for Everything

E5 — это семейство моделей для создания текстовых эмбеддингов от Microsoft, основанное на энкодерах T5 и обученное с использованием contrastive learning на большом наборе данных. Это позволяет генерировать универсальные эмбеддинги для различных задач.

### Контекст

Создание универсальных текстовых эмбеддингов для задач, таких как семантический поиск и кластеризация, долгое время было целью. Существующие модели часто требовали дообучения или были проприетарными (например, OpenAI Embeddings, 2022). E5 стремится стать открытой и мощной моделью для "эмбеддингов для всего".

### Идея

E5 предлагает единую модель для получения dense embeddings, превосходящую специализированные модели. Основные инновации:
1. **Энкодер T5:** Использование Transformer-энкодера T5.
2. **Масштабные данные:** Обучение на большом наборе данных с миллионами пар (запрос, документ).
3. **Методика обучения:** Contrastive learning с форматированием входных данных и mean pooling для агрегации.

### Постановка задачи

E5 создает dense vector representations для текстов, чтобы семантически похожие тексты имели близкие эмбеддинги. Эти эмбеддинги используются для поиска и сравнения текстов.

### Альтернативные методы

На момент появления E5 существовали:
* **Sparse Retrieval (BM25):** Основан на частоте слов, плохо улавливает семантику.
* **Sentence-BERT (SBERT, 2019):** Дообучение BERT для эмбеддингов предложений.
* **Dense Passage Retrieval (DPR, 2020):** Двухбашенная архитектура на базе BERT.
* **Проприетарные модели:** Закрытые, высокопроизводительные решения.

### Архитектура

E5 использует **энкодер T5** различных размеров: `e5-small`, `e5-base`, `e5-large`, `e5-xl`, `e5-2b`. Основные компоненты:
1. **Входное форматирование:** Префиксы `query:` и `passage:` для различения ролей текста.
2. **T5 Encoder:** Преобразование текста через Transformer-энкодер T5.
3. **Pooling Layer:** Mean pooling для агрегации токенов.
4. **L2 Normalization:** Нормализация эмбеддингов.

### Алгоритм обучения

Обучение E5 основано на **contrastive learning**:
1. **Набор данных:** Используются пары (запрос, релевантный документ) из множества источников.
2. **Формирование батчей:** Использование in-batch и hard negatives.
3. **Функция потерь:** InfoNCE loss для максимизации сходства положительных пар.
4. **Шаги обучения:** Преобразование текста, пропуск через энкодер, mean pooling, нормализация и обновление весов.

### Алгоритм инференса

1. **Эмбеддинг запроса:** Префикс `query:`, пропуск через энкодер, mean pooling, нормализация.
2. **Эмбеддинг документа:** Префикс `passage:`, аналогично запросу.
3. **Поиск сходства:** Использование ANN-индекса для поиска ближайших эмбеддингов.

<img src="img/img.png" width=500>

### Результаты

E5 показала выдающиеся результаты:
* **MTEB Leaderboard:** `e5-large` и `e5-2b` заняли первые места на Massive Text Embedding Benchmark.
* **Производительность Retrieval:** Превосходство на MS MARCO и BEIR по nDCG@K и Recall@K.
* **Обобщающая способность:** Высокая производительность без дообучения, подтверждая концепцию "эмбеддингов для всего".
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример использования модели E5 для генерации текстовых эмбеддингов
# Мы будем использовать библиотеку Hugging Face Transformers, которая предоставляет доступ к модели E5.

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

# Загрузка токенизатора и модели E5
tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-base")
model = AutoModel.from_pretrained("intfloat/e5-base")

# Функция для генерации эмбеддингов
def generate_embedding(text, prefix):
    # Добавляем префикс к тексту
    formatted_text = f"{prefix}: {text}"
    
    # Токенизация текста
    inputs = tokenizer(formatted_text, return_tensors="pt", padding=True, truncation=True)
    
    # Пропуск текста через модель
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Применение mean pooling к выходным скрытым состояниям
    embeddings = outputs.last_hidden_state.mean(dim=1)
    
    # L2-нормализация эмбеддингов
    normalized_embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    
    return normalized_embeddings

# Пример запроса и документа
query = "What is artificial intelligence?"
document = "Artificial intelligence is the simulation of human intelligence processes by machines."

# Генерация эмбеддингов для запроса и документа
query_embedding = generate_embedding(query, "query")
document_embedding = generate_embedding(document, "passage")

# Вычисление косинусного сходства между эмбеддингами
cosine_similarity = torch.nn.functional.cosine_similarity(query_embedding, document_embedding)

print(f"Cosine Similarity between query and document: {cosine_similarity.item()}")

# Пример использования FAISS для поиска ближайших соседей
import faiss

# Создание индекса FAISS
dimension = query_embedding.shape[1]  # Размерность эмбеддингов
index = faiss.IndexFlatL2(dimension)  # Используем L2 расстояние

# Добавление эмбеддингов документов в индекс
# В реальном сценарии вы бы добавили эмбеддинги всех документов в корпусе
index.add(document_embedding.numpy())

# Поиск ближайших соседей для эмбеддинга запроса
k = 1  # Количество ближайших соседей
D, I = index.search(query_embedding.numpy(), k)

print(f"Indices of nearest neighbors: {I}")
print(f"Distances to nearest neighbors: {D}")
```

### Объяснение ключевых моментов:

1. **Форматирование входных данных:** Мы добавляем префиксы `query:` и `passage:` к текстам, чтобы модель могла различать их роли. Это специфическая методика, предложенная в E5, которая помогает модели лучше понимать контекст.

2. **Mean Pooling:** Вместо использования `[CLS]` токена, мы применяем mean pooling к выходным скрытым состояниям, что позволяет агрегировать информацию из всех токенов. Это часто дает более стабильные и общие эмбеддинги.

3. **L2-нормализация:** Нормализуем эмбеддинги, что является стандартной практикой для dense embeddings, особенно при использовании косинусного сходства.

4. **Использование FAISS:** Мы используем FAISS для поиска ближайших соседей, что позволяет эффективно находить документы, наиболее похожие на запрос.

Этот пример демонстрирует, как можно использовать модель E5 для генерации универсальных текстовых эмбеддингов и выполнять задачи семантического поиска.